# Research notebook

Read the module README before executing. Textual outputs below are saved historical snapshots; the report follows the original project document and was not recalculated during publication cleanup. Setup paths have been made portable. Embedded media and machine-specific diagnostic output are omitted from this public-facing copy.


In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import PICTURE_DIR, GALLERY_DIR, OUTPUT_DIR, LLAVA_MODEL, TARGET_CLASSES

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [1]:
import os
from PIL import Image
import torch
import clip
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
import numpy as np
from pathlib import Path
def load_data(folder_path):
    images = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            with Image.open(file_path) as img:
                images.append(img.copy())
        except Exception as e:
            print(f"无法打开文件 {filename}: {e}")
    return images


picture = list(TARGET_CLASSES)

In [2]:
print(device)

cuda


In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 确保模型在GPU上
model.to(device)

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [4]:
def subfolder(exclude):
    root = PICTURE_DIR
    return [str(item) for subdir in root.iterdir()
            if subdir.is_dir()
            for item in subdir.iterdir()
            if item.is_dir() and item!=(PICTURE_DIR / exclude / exclude)]
def create_path(figure):
    positive_path=str(PICTURE_DIR / figure / figure)
    negative_path=subfolder(figure)
    return positive_path,negative_path
def load_images(folder_path):
    """原函数：一次性加载所有图片（保留原逻辑）"""
    images = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            with Image.open(file_path) as img:
                images.append(img.copy())
        except Exception as e:
            print(f"无法打开文件 {filename}: {e}")
    return images
def create_data(p,n):
    P=load_images(p)
    N=[]
    for a in n:
        b=load_images(a)
        N=N+b
    return P,N
def prediction(data, y, batch_size=200):

    text = clip.tokenize([y, 'others']).to(device)
    score = []
    for i in range(0, len(data), batch_size):
        batch_data = data[i:i+batch_size]


        batch_images = []
        for img in batch_data:
            processed_img = preprocess(img).unsqueeze(0)  # 增加批次维度
            batch_images.append(processed_img)

        batch_tensor = torch.cat(batch_images, dim=0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(batch_tensor)
            text_features = model.encode_text(text)
            image_features = image_features / image_features.norm(dim=1, keepdim=True)
            text_features = text_features / text_features.norm(dim=1, keepdim=True)
            logits_per_image = model.logit_scale.exp() * image_features @ text_features.t()
            batch_probs = logits_per_image.softmax(dim=-1).cpu().numpy()[:, 0]
            score.extend(batch_probs)

    return score
def score(figure):
    positive_path,negative_path=create_path(figure)
    positive_data,negative_data=create_data(positive_path,negative_path)
    Pscore=prediction(positive_data,figure)
    Nscore=prediction(negative_data,figure)
    return Pscore,Nscore
def evaluation(threshold,Pscore,Nscore):
    Pclass=[1 if x>=threshold else 0 for x in Pscore]
    Nclass=[1 if x>=threshold else 0 for x in Nscore]
    TP=Pclass.count(1)
    FN=Pclass.count(0)
    FP=Nclass.count(1)
    if TP+FP!=0:
        Precision=TP/(TP+FP)
    if TP+FP==0:
        Precision=0
    if TP+FN!=0:
        Recall=TP/(TP+FN)
    if TP+FN==0:
        Recall=0
    return Precision, Recall

In [5]:

Pscore={k:[] for k in picture}
Nscore={k:[] for k in picture}

for figure,x in zip(picture,range(len(picture))):
    print(figure)
    pscore,nscore=score(figure)
    Pscore[figure]=Pscore[figure]+pscore
    Nscore[figure]=Nscore[figure]+nscore



Dog


Piano
Erhu
Porcelain
Duck


In [8]:
print(Pscore)

{'Dog': [np.float16(0.9995), np.float16(0.9985), np.float16(0.9976), np.float16(0.9995), np.float16(0.992), np.float16(0.999), np.float16(0.998), np.float16(0.562), np.float16(0.997), np.float16(0.999), np.float16(0.9995), np.float16(0.994), np.float16(0.9946), np.float16(0.9985), np.float16(0.999), np.float16(0.999), np.float16(0.994), np.float16(0.999), np.float16(0.985), np.float16(0.994), np.float16(0.9956), np.float16(0.9995), np.float16(0.9897), np.float16(0.9995), np.float16(0.9995), np.float16(0.969), np.float16(0.9995), np.float16(1.0), np.float16(0.9995), np.float16(0.9937), np.float16(0.998), np.float16(0.998), np.float16(0.9976), np.float16(0.9995), np.float16(0.999), np.float16(0.9937), np.float16(0.999), np.float16(0.9985), np.float16(0.999), np.float16(0.9995), np.float16(1.0), np.float16(0.999), np.float16(0.9727), np.float16(0.9424), np.float16(0.999), np.float16(0.998), np.float16(0.9854), np.float16(0.9966), np.float16(0.9985), np.float16(0.9775), np.float16(0.9976),

In [9]:
print(Nscore)

{'Dog': [np.float16(0.785), np.float16(0.8887), np.float16(0.766), np.float16(0.7373), np.float16(0.813), np.float16(0.746), np.float16(0.3667), np.float16(0.6406), np.float16(0.7188), np.float16(0.699), np.float16(0.5117), np.float16(0.746), np.float16(0.743), np.float16(0.9004), np.float16(0.693), np.float16(0.772), np.float16(0.758), np.float16(0.461), np.float16(0.8843), np.float16(0.923), np.float16(0.558), np.float16(0.793), np.float16(0.2338), np.float16(0.813), np.float16(0.725), np.float16(0.842), np.float16(0.566), np.float16(0.8174), np.float16(0.4072), np.float16(0.1294), np.float16(0.879), np.float16(0.731), np.float16(0.562), np.float16(0.86), np.float16(0.8354), np.float16(0.504), np.float16(0.8154), np.float16(0.755), np.float16(0.5737), np.float16(0.808), np.float16(0.9033), np.float16(0.696), np.float16(0.709), np.float16(0.947), np.float16(0.094), np.float16(0.6826), np.float16(0.5776), np.float16(0.504), np.float16(0.5776), np.float16(0.3486), np.float16(0.699), np.

In [6]:
# 不卡阈值
f1_score0 = {k: 0 for k in picture}
for figure, x in zip(picture, range(len(picture))):
    precision, recall = evaluation(0.5, Pscore[figure], Nscore[figure])
    if precision + recall == 0:
        f1_score0[figure] = 0
    else:
        f = 2 * (precision * recall) / (precision + recall)
        f1_score0[figure] = f
        print(f'{figure}: precision:{precision:.4f},recall:{recall:.4f},f1:{f:.4f}')
print(f1_score0)

Dog: precision:0.2307,recall:1.0000,f1:0.3749
Piano: precision:0.2608,recall:1.0000,f1:0.4137
Erhu: precision:0.1784,recall:1.0000,f1:0.3028
Porcelain: precision:0.3053,recall:1.0000,f1:0.4678
Duck: precision:0.2448,recall:1.0000,f1:0.3933
{'Dog': 0.3748828491096532, 'Piano': 0.4136504653567736, 'Erhu': 0.3028009084027252, 'Porcelain': 0.4678362573099415, 'Duck': 0.39331366764995085}


In [10]:
# 每个类别卡阈值
Threshold0 = {k: 0 for k in picture}
f1_score = {k: 0 for k in picture}

for figure, x in zip(picture, range(len(picture))):
    pre = 0
    rec = 0
    for threshold in np.arange(0.0, 1.01, 0.01):
        precision, recall = evaluation(threshold, Pscore[figure], Nscore[figure])
        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * (precision * recall) / (precision + recall)
        if f1 >= f1_score[figure]:
            Threshold0[figure] = threshold
            f1_score[figure] = f1
            pre = precision
            rec = recall
    print(f'{figure},precision:{pre:.4f},recall:{rec:.4f},f1:{f1_score[figure]:.4f},best_threshold:{Threshold0[figure]}')


Dog,precision:0.8685,recall:0.9250,f1:0.8959,best_threshold:0.97
Piano,precision:0.6277,recall:0.8600,f1:0.7257,best_threshold:1.0
Erhu,precision:0.8102,recall:0.8750,f1:0.8413,best_threshold:1.0
Porcelain,precision:0.7072,recall:0.9300,f1:0.8035,best_threshold:0.92
Duck,precision:0.7100,recall:0.9550,f1:0.8145,best_threshold:0.98


In [11]:
# 卡整体阈值

max_macro_f1 = 0
Threshold = 0
for threshold in np.arange(0.0, 1.01, 0.01):
    f1_score = {k: 0 for k in picture}
    for figure, x in zip(picture, range(len(picture))):
        precision, recall = evaluation(threshold, Pscore[figure], Nscore[figure])
        if precision + recall == 0:
            f1_score[figure] = 0
        else:
            f1_score[figure] = 2 * (precision * recall) / (precision + recall)
    macro_f1 = sum([value for value in f1_score.values()]) / len(f1_score)
    if macro_f1 >= max_macro_f1:
        Threshold = threshold
        max_macro_f1 = macro_f1

print(Threshold, max_macro_f1)

0.98 0.738397818851723


# 合并+去重

In [13]:
print(Pscore)

{'Dog': [np.float16(0.9995), np.float16(0.9985), np.float16(0.9976), np.float16(0.9995), np.float16(0.992), np.float16(0.999), np.float16(0.998), np.float16(0.562), np.float16(0.997), np.float16(0.999), np.float16(0.9995), np.float16(0.994), np.float16(0.9946), np.float16(0.9985), np.float16(0.999), np.float16(0.999), np.float16(0.994), np.float16(0.999), np.float16(0.985), np.float16(0.994), np.float16(0.9956), np.float16(0.9995), np.float16(0.9897), np.float16(0.9995), np.float16(0.9995), np.float16(0.969), np.float16(0.9995), np.float16(1.0), np.float16(0.9995), np.float16(0.9937), np.float16(0.998), np.float16(0.998), np.float16(0.9976), np.float16(0.9995), np.float16(0.999), np.float16(0.9937), np.float16(0.999), np.float16(0.9985), np.float16(0.999), np.float16(0.9995), np.float16(1.0), np.float16(0.999), np.float16(0.9727), np.float16(0.9424), np.float16(0.999), np.float16(0.998), np.float16(0.9854), np.float16(0.9966), np.float16(0.9985), np.float16(0.9775), np.float16(0.9976),

In [14]:
Pclass={k:[] for k in picture}
Nclass={k:[] for k in picture}
for x in picture:
    print(x,Threshold0[x])
    pclass=[1 if y>=Threshold0[x] else 0 for y in Pscore[x]]
    nclass=[1 if y>=Threshold0[x] else 0 for y in Nscore[x]]
    Pclass[x]=pclass
    Nclass[x]=nclass


Dog 0.97
Piano 1.0
Erhu 1.0
Porcelain 0.92
Duck 0.98


In [15]:
import pickle
import pandas as pd
with open(OUTPUT_DIR / '英文Clip每类正样本预测结果.pkl', 'wb') as f:
    pickle.dump(Pclass, f)
with open(OUTPUT_DIR / '英文Clip每类负样本预测结果.pkl', 'wb') as f:
    pickle.dump(Nclass, f)
